# Silver Layer — Orders

## Objective

Transform and validate the orders data from the Bronze layer.

## Data Quality Checks

- Missing delivery date for delivered orders
- Inconsistent delivery dates
- Duplicate order IDs
- Null values on each column

In [0]:
%sql
DESCRIBE TABLE workspace.bronze.olist_orders_dataset

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver

In [0]:
%sql
--- Verifying duplicates on order_id
SELECT
    order_id,
    COUNT(*) AS quantidade
FROM workspace.bronze.olist_orders_dataset
GROUP BY order_id
HAVING COUNT(*) > 1;

In [0]:
%sql
--- Verifying NULL values on each column
SELECT
    COUNT(*) AS total,
    COUNT(order_approved_at) AS approved,
    COUNT(order_delivered_carrier_date) AS carrier,
    COUNT(order_delivered_customer_date) AS delivered,
    COUNT(order_estimated_delivery_date) AS estimated
FROM workspace.bronze.olist_orders_dataset;

In [0]:
%sql
--- Data Quality: Crossing order status with order delivery date
SELECT
    order_status,
    COUNT(*) AS total,
    COUNT(order_delivered_customer_date) AS delivered,
    COUNT(*) - COUNT(order_delivered_customer_date) AS null_delivered
FROM workspace.bronze.olist_orders_dataset
GROUP BY order_status
ORDER BY total DESC;

In [0]:
%sql
--- Analyzing the NULL values on order_delivered_customer_date if status = 'delivered'
SELECT *
FROM workspace.bronze.olist_orders_dataset
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NULL;

In [0]:
%sql
SELECT COUNT(*) AS inconsistentes
FROM workspace.bronze.olist_orders_dataset
WHERE order_delivered_customer_date < order_delivered_carrier_date;

In [0]:
%sql
SELECT
    order_id,
    order_status,
    order_delivered_carrier_date,
    order_delivered_customer_date
FROM workspace.bronze.olist_orders_dataset
WHERE order_delivered_customer_date < order_delivered_carrier_date;

In [0]:
%sql
SELECT
    order_status,
    COUNT(*) AS casos
FROM workspace.bronze.olist_orders_dataset
WHERE order_delivered_customer_date IS NOT NULL
GROUP BY order_status
ORDER BY casos DESC;

In [0]:
%sql
CREATE TABLE workspace.silver.olist_orders_dataset
USING delta
AS
SELECT 
    order_id,
    customer_id,
    order_status,

    order_purchase_timestamp,
    CAST(order_purchase_timestamp AS DATE) AS purchase_date,

    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date,

    CASE
        WHEN order_status = 'delivered'
            AND order_delivered_customer_date IS NULL
        THEN TRUE
        ELSE FALSE
    END AS delivery_date_inconsistent,

    CASE
        WHEN order_delivered_customer_date < order_delivered_carrier_date
        THEN TRUE
        ELSE FALSE
    END AS delivery_dates_inconsistent

FROM workspace.bronze.olist_orders_dataset;


# Silver Layer — Customers

## Objective

Transform and validate the customers data from the Bronze layer.

## Data Quality Checks
- Duplicate customer IDs
- Inconsistencies in customer_state
- Difference between customer_id and customer_unique_id

In [0]:
%sql
DESCRIBE TABLE workspace.silver.olist_customers_dataset

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS unique_customer_ids,
    COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM workspace.bronze.olist_customers_dataset;

In [0]:
%sql
--- Verifying what customer_id means
SELECT
    c.customer_unique_id,
    COUNT(*) AS customer_records,
    COUNT(DISTINCT o.order_id) AS orders
FROM workspace.bronze.olist_customers_dataset c
JOIN workspace.bronze.olist_orders_dataset o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_unique_id
HAVING COUNT(*) > 1
ORDER BY orders DESC
LIMIT 10;

Each order is associated with a unique `customer_id`, while `customer_unique_id` identifies the customer across multiple orders.

In [0]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(customer_id) AS customer_id,
    COUNT(customer_unique_id) AS customer_unique_id,
    COUNT(customer_zip_code_prefix) AS zip_code,
    COUNT(customer_city) AS city,
    COUNT(customer_state) AS state
FROM workspace.bronze.olist_customers_dataset;

In [0]:
%sql
SELECT
    customer_state,
    COUNT(*) AS customers
FROM workspace.bronze.olist_customers_dataset
GROUP BY customer_state
ORDER BY customers DESC;

In [0]:
%sql
SELECT
    MIN(customer_zip_code_prefix) AS min_zip,
    MAX(customer_zip_code_prefix) AS max_zip
FROM workspace.bronze.olist_customers_dataset;

In [0]:
%sql

CREATE TABLE workspace.silver.olist_customers_dataset
USING DELTA
AS
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM workspace.bronze.olist_customers_dataset;

# Silver Layer — Geolocation

## Data Quality Checks

- Checked for missing values in ZIP code, latitude, longitude, city, and state.
- Validated that latitude and longitude values are within valid geographic ranges.
- Confirmed that the ZIP code prefix is not unique, as multiple geographic records can be associated with the same prefix.

## Transformation

No data transformation was required at this stage. The validated Bronze data will be materialized as a Silver Delta table.

In [0]:
%sql
DESCRIBE TABLE workspace.bronze.olist_geolocation_dataset

In [0]:
%sql
--- Verifying the limits on latitude and longitude. latitude has to be between -90 and 90, while longitude has to be between -180 and 180
SELECT 
    MAX(geolocation_lat) AS max_lat,
    MIN(geolocation_lat) AS min_lat,
    MAX(geolocation_lng) AS max_lng,
    MIN(geolocation_lng) AS min_lng
FROM workspace.bronze.olist_geolocation_dataset;

In [0]:
%sql
SELECT DISTINCT(geolocation_state)
FROM workspace.bronze.olist_geolocation_dataset; 

In [0]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(geolocation_zip_code_prefix) AS zip_code,
    COUNT(geolocation_lat) AS latitude,
    COUNT(geolocation_lng) AS longitude,
    COUNT(geolocation_city) AS city,
    COUNT(geolocation_state) AS state
FROM workspace.bronze.olist_geolocation_dataset;

In [0]:
%sql
CREATE TABLE workspace.silver.olist_geolocation_dataset
USING DELTA
AS
SELECT
    CAST(geolocation_zip_code_prefix AS STRING) AS geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state
FROM workspace.bronze.olist_geolocation_dataset;


# Silver Layer — Order Items

## Objective

Validate and prepare the order items data from the Bronze layer.

## Data Quality Checks

The table will be analyzed to verify its granularity, key uniqueness, missing values, and consistency of numeric and date fields.

In [0]:
%sql

DESCRIBE TABLE workspace.bronze.olist_order_items_dataset;

In [0]:
%sql
--- Verifying duplicates
SELECT 
    COUNT(*),
    COUNT(DISTINCT CONCAT(order_id, '_', order_item_id)) AS unique_items
FROM workspace.bronze.olist_order_items_dataset;

In [0]:
%sql
--- Verifying NULL values
SELECT
    COUNT(*) AS total,
    COUNT(order_id) AS order_id,
    COUNT(order_item_id) AS order_item_id,
    COUNT(product_id) AS product_id,
    COUNT(seller_id) AS seller_id,
    COUNT(shipping_limit_date) AS shipping_limit_date,
    COUNT(price) AS price,
    COUNT(freight_value) AS freight_value
FROM workspace.bronze.olist_order_items_dataset;

In [0]:
%sql
SELECT
    MIN(price) AS min_price,
    MAX(price) AS max_price,
    MIN(freight_value) AS min_freight,
    MAX(freight_value) AS max_freight
FROM workspace.bronze.olist_order_items_dataset;

In [0]:
%sql
--- Verifying shipping date limits. Searching for unfamiliar shipping date.
SELECT
    MIN(shipping_limit_date) AS min_shipping_limit,
    MAX(shipping_limit_date) AS max_shipping_limit
FROM workspace.bronze.olist_order_items_dataset;

In [0]:
%sql
SELECT
    oi.order_id,
    oi.order_item_id,
    o.order_purchase_timestamp,
    oi.shipping_limit_date
FROM workspace.bronze.olist_order_items_dataset oi
JOIN workspace.bronze.olist_orders_dataset o
    ON oi.order_id = o.order_id
WHERE oi.shipping_limit_date < o.order_purchase_timestamp;

In [0]:
%sql
CREATE TABLE workspace.silver.olist_order_items_dataset
USING DELTA
AS
SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
FROM workspace.bronze.olist_order_items_dataset;

# Silver Layer — Order Reviews

In [0]:
%sql

DESCRIBE workspace.bronze.olist_order_reviews_dataset;

In [0]:
%sql
SELECT *
FROM workspace.bronze.olist_order_reviews_dataset
LIMIT 20;

In [0]:
%sql
SELECT
    review_id,
    COUNT(*) AS occurrences
FROM workspace.bronze.olist_order_reviews_dataset
GROUP BY review_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 20;

In [0]:
%sql
SELECT *
FROM workspace.bronze.olist_order_reviews_dataset
WHERE review_id = 'dbdf1ea31790c8ecfcc6750525661a9b';

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(review_id, '_', order_id)) AS unique_review_order
FROM workspace.bronze.olist_order_reviews_dataset;

In [0]:
%sql
CREATE TABLE workspace.silver.olist_order_reviews_dataset
USING DELTA
AS
SELECT
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
FROM workspace.bronze.olist_order_reviews_dataset;


# Silver Layer — Order Payments

## Objective

Validate and prepare the order payment data from the Bronze layer.

## Data Quality Checks

The table will be analyzed to verify its granularity, key uniqueness, missing values, and consistency of numeric and date fields.

In [0]:
%sql
SELECT *
FROM workspace.bronze.olist_order_payments_dataset
LIMIT 10;

In [0]:
%sql
DESCRIBE TABLE workspace.bronze.olist_order_payments_dataset;

In [0]:
%sql
CREATE TABLE workspace.silver.olist_order_payments_dataset
USING DELTA
AS
SELECT
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
FROM workspace.bronze.olist_order_payments_dataset;


# Silver Layer — Products

In [0]:
%sql
DESCRIBE workspace.bronze.olist_products_dataset;

In [0]:
%sql
SELECT *
FROM workspace.bronze.olist_products_dataset
LIMIT 10;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(product_category_name) AS n_products,
    COUNT(product_name_lenght) AS n_products_with_name,
    COUNT(product_description_lenght) AS n_products_with_description,
    COUNT(product_photos_qty) AS n_products_with_photos,
    COUNT(product_weight_g) AS n_products_with_weight,
    COUNT(product_length_cm) AS n_products_with_length,
    COUNT(product_height_cm) AS n_products_with_height,
    COUNT(product_width_cm) AS n_products_with_width,
    COUNT(product_weight_g) AS n_products_with_weight,
    COUNT(product_photos_qty) AS n_products_with_photos
FROM workspace.bronze.olist_products_dataset

In [0]:
%sql
CREATE TABLE workspace.silver.olist_products_dataset
USING DELTA
AS
SELECT
    product_id,
    product_category_name,
    product_name_lenght,
    product_description_lenght,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM workspace.bronze.olist_products_dataset;

# Silver Layer — Sellers

In [0]:
%sql
DESCRIBE workspace.bronze.olist_sellers_dataset;

In [0]:
%sql
SELECT *
FROM workspace.bronze.olist_sellers_dataset
LIMIT 10;

In [0]:
%sql
CREATE TABLE workspace.silver.olist_sellers_dataset
USING DELTA
AS
SELECT
    seller_id,
    CAST(seller_zip_code_prefix AS STRING) AS seller_zip_code_prefix,
    seller_city,
    seller_state
FROM workspace.bronze.olist_sellers_dataset;

# Silver Layer — Product Category Translation

In [0]:
%sql
DESCRIBE workspace.bronze.product_category_name_translation;

In [0]:
%sql
SELECT *
FROM workspace.bronze.product_category_name_translation
LIMIT 10;

In [0]:
%sql
CREATE TABLE workspace.silver.product_category_name_translation
USING DELTA
AS
SELECT
    product_category_name,
    product_category_name_english
FROM workspace.bronze.product_category_name_translation;